In [ ]:
!pip install datasets transformers pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 9.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing i

# CONFIG


In [ ]:
import torch

class Config:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_samples = 500
    test_samples = 200
    batch_size = 4

    n_qubits = 6
    K = 2
    T = 2
    n_layers = 4

    epochs = 10
    lr = 1e-3   # 🔥 increased

cfg = Config()

# DATA PIPELINE


In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

dataset = load_dataset("gsm8k", "main")

train_data = [dataset["train"][i] for i in range(cfg.train_samples)]
test_data = [dataset["test"][i] for i in range(cfg.test_samples)]

def extract_label(example):
    match = re.search(r"#### (\d+)", example["answer"])
    if match:
        return min(int(match.group(1)) // 10, 99)
    return 0

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased").to(cfg.device)
encoder.eval()

def encode_batch(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(cfg.device)
    with torch.no_grad():
        outputs = encoder(**inputs)
    return outputs.last_hidden_state[:, 0, :]

def build_encoded_dataset(data):
    encoded = []
    for i in tqdm(range(0, len(data), 32), desc="Encoding"):
        batch = data[i:i+32]
        texts = [x["question"] for x in batch]
        labels = [extract_label(x) for x in batch]

        emb = encode_batch(texts).cpu()

        for j in range(len(batch)):
            encoded.append((emb[j], labels[j]))
    return encoded

train_encoded = build_encoded_dataset(train_data)
test_encoded = build_encoded_dataset(test_data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding: 100%|██████████| 7/7 [00:00<00:00, 11.53it/s]


# QUANTUM NODE


In [ ]:
import pennylane as qml
import torch.nn as nn
import math

try:
    dev = qml.device("lightning.gpu", wires=cfg.n_qubits)
    print("⚡ Using lightning.gpu")
except:
    dev = qml.device("default.qubit", wires=cfg.n_qubits)
    print("⚠️ Using default.qubit")

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qnode_batch(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(cfg.n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(cfg.n_qubits))
    return tuple(qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits))

⚡ Using lightning.gpu


# CLASSICAL BASELINE


In [ ]:
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 100)
        )

    def forward(self, x):
        return self.net(x)

# qAIR v10 MODEL


In [ ]:
class QuantumV11(nn.Module):
    def __init__(self):
        super().__init__()

        self.K = cfg.K
        self.T = cfg.T

        self.proj = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, cfg.n_qubits)
        )

        self.step_embed = nn.Embedding(cfg.T, cfg.n_qubits)

        self.q_weights = nn.Parameter(
            0.01 * torch.randn(cfg.n_layers, cfg.n_qubits, 3)
        )

        self.refine = nn.Sequential(
            nn.Linear(cfg.n_qubits, cfg.n_qubits),
            nn.Tanh()
        )

        self.score = nn.Sequential(
            nn.Linear(cfg.n_qubits, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.decoder = nn.Sequential(
            nn.Linear(cfg.n_qubits, 128),
            nn.ReLU(),
            nn.Linear(128, 100)
        )

    def forward(self, x, return_all=False):

        B = x.shape[0]

        base = torch.tanh(self.proj(x)) * math.pi

        # 🔥 STRONG DIVERSITY INITIALIZATION
        state = torch.stack([
            base * (1 + 0.3 * i) + 0.5 * torch.randn_like(base)
            for i in range(self.K)
        ], dim=1)

        all_states = []

        for t in range(self.T):

            step_emb = self.step_embed(torch.tensor(t, device=x.device))
            state = state + step_emb

            # 🔥 REPULSIVE ENTANGLEMENT (CRITICAL)
            diff = state.unsqueeze(2) - state.unsqueeze(1)
            repulsion = diff.mean(dim=2)
            state = state + 0.3 * repulsion

            # 🔥 QUANTUM ONLY AT FIRST STEP
            if t == 0:
                s_all = state.view(-1, cfg.n_qubits)

                q_out = qnode_batch(s_all, self.q_weights)
                q_out = torch.stack(q_out, dim=-1)

                q_out = q_out.to(x.device).float()

                # 🔥 NORMALIZATION (STABILITY)
                q_out = q_out / (torch.norm(q_out, dim=-1, keepdim=True) + 1e-6)

                q_out = q_out.view(B, self.K, cfg.n_qubits)

                state = state + 0.5 * self.refine(q_out)

            else:
                state = state + 0.3 * self.refine(state)

            state = torch.clamp(state, -math.pi, math.pi)

            all_states.append(state)

        # 🔥 STRONGER COLLAPSE
        scores = self.score(state).squeeze(-1)
        scores = scores + torch.norm(state, dim=-1)

        weights = torch.softmax(scores, dim=1)

        collapsed = (state * weights.unsqueeze(-1)).sum(dim=1)

        logits = self.decoder(collapsed)

        if return_all:
            return logits, state, all_states, weights

        return logits

# TRAIN LOOP


In [ ]:
from torch.cuda.amp import autocast, GradScaler
import torch.nn as nn

def build_loader(data):
    for i in range(0, len(data), cfg.batch_size):
        batch = data[i:i+cfg.batch_size]

        x = torch.stack([b[0] for b in batch]).to(cfg.device)
        y = torch.tensor([b[1] for b in batch]).to(cfg.device)

        yield x, y


def train(model, data):

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    scaler = GradScaler()
    loss_fn = nn.CrossEntropyLoss()

    is_quantum = hasattr(model, "K")

    for epoch in range(cfg.epochs):

        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for xb, yb in build_loader(data):

            with autocast():

                if is_quantum:
                    logits, state, _, _ = model(xb, return_all=True)

                    ce_loss = loss_fn(logits, yb)
                    div_loss = -torch.mean(torch.var(state, dim=1))

                    loss = ce_loss + 0.1 * div_loss
                else:
                    logits = model(xb)
                    loss = loss_fn(logits, yb)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)

        print(f"Epoch {epoch+1}: Loss={total_loss:.4f}, Acc={(correct/total)*100:.2f}%")

# EVALUATION


In [ ]:
import time
from sklearn.metrics import confusion_matrix

def evaluate(model, data):

    model.eval()

    correct = 0
    total = 0
    best_of_k = 0

    all_preds = []
    all_labels = []

    is_quantum = hasattr(model, "K")

    start = time.time()

    with torch.no_grad():
        for xb, yb in build_loader(data):

            if is_quantum:
                logits, state, _, _ = model(xb, return_all=True)
            else:
                logits = model(xb)
                state = None

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()

            if is_quantum:
                for k in range(model.K):
                    logits_k = model.decoder(state[:, k, :])
                    preds_k = logits_k.argmax(dim=1)
                    best_of_k += (preds_k == yb).sum().item()

            total += len(yb)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    end = time.time()

    acc = correct / total
    best_k = best_of_k / (total * model.K) if is_quantum else None
    tps = (end - start) / total

    print("\n📊 Evaluation")
    print(f"Accuracy: {acc*100:.2f}%")
    if best_k is not None:
        print(f"Best-of-K: {best_k*100:.2f}%")
    print(f"Time/sample: {tps:.4f}s")

    return acc, best_k

# RUN


In [ ]:
models = {
    "Classical": ClassicalModel().to(cfg.device),
    "Quantum v11": QuantumV11().to(cfg.device)
}

for name, model in models.items():

    print(f"\n🚀 Training {name}")
    train(model, train_encoded)

    evaluate(model, test_encoded)


🚀 Training Classical


/tmp/ipykernel_1159/18374207.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1159/18374207.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Loss=424.7896, Acc=16.80%
Epoch 2: Loss=378.8651, Acc=18.60%
Epoch 3: Loss=367.6511, Acc=20.60%
Epoch 4: Loss=357.0165, Acc=24.20%
Epoch 5: Loss=346.1254, Acc=24.60%
Epoch 6: Loss=336.3284, Acc=25.20%
Epoch 7: Loss=325.2602, Acc=28.40%
Epoch 8: Loss=313.6113, Acc=30.00%
Epoch 9: Loss=301.3105, Acc=32.40%
Epoch 10: Loss=288.7650, Acc=34.40%

📊 Evaluation
Accuracy: 17.50%
Time/sample: 0.0002s

🚀 Training Quantum v11
Epoch 1: Loss=420.5892, Acc=17.80%
Epoch 2: Loss=387.2649, Acc=19.20%
Epoch 3: Loss=380.0770, Acc=17.20%
Epoch 4: Loss=374.9336, Acc=18.60%
Epoch 5: Loss=372.9467, Acc=18.60%
Epoch 6: Loss=372.3400, Acc=19.40%
Epoch 7: Loss=367.3723, Acc=19.60%
Epoch 8: Loss=363.8315, Acc=20.60%
Epoch 9: Loss=359.6426, Acc=21.80%
Epoch 10: Loss=356.8217, Acc=22.80%

📊 Evaluation
Accuracy: 19.50%
Best-of-K: 18.00%
Time/sample: 0.0877s
